# Matrix completion via recommendation system example

This example demonstrates the use of matrix completion techniques on a recommendation system.  The recommendation system uses data from the [360K Last.fm dataset](http://ocelma.net/MusicRecommendationDataset/lastfm-360K.html).

In [2]:
!pip install -U implicit h5py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.4/761.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 10.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: h5py
    Found existing installation: h5py 3.13.0
    Uninstalling h5py-3.13.0:
      Successfully uninstalled h5py-3.13.0


In [3]:
# retrieving last.fm dataset
from implicit.datasets.lastfm import get_lastfm
import numpy as np
import pandas as pd
from scipy import sparse
import os
from pathlib import Path

## Downloading and saving the Last.fm dataset

In [4]:
filepath = r'datasets/'
Path(filepath).mkdir(exist_ok=True)

if not os.path.exists(filepath + r'artist_user_plays.npz'):
    # save our dataset in sparse format
    artists, users, artist_user_plays = get_lastfm()

    sparse.save_npz(filepath + r'artist_user_plays.npz', artist_user_plays)
    np.save(filepath + 'artists.npy', artists)
    np.save(filepath + 'users.npy', users)
else:
    # load our dataset into original format
    artist_user_plays = sparse.load_npz(filepath + r'artist_user_plays.npz')
    artists = np.load(filepath + 'artists.npy', allow_pickle=True)
    users = np.load(filepath + 'users.npy', allow_pickle=True)

0.00B [00:00, ?B/s]

In [ ]:
# investigate the content of the downloaded dataset
artists#.shape()
users#.shape()

array(['00000c289a1829a808ac09c00daf10bc3c4e223b',
       '00001411dc427966b17297bf4d69e7e193135d89',
       '00004d2ac9316e22dc007ab2243d6fcb239e707d', ...,
       'ffff9af9ae04d263dae91cb838b1f3a6725f5ffb',
       'ffff9ef87a7d9494ada2f9ade4b9ff637c0759ac', 'sep 20, 2008'],
      dtype=object)

In [9]:
# return the dimensions of data
artists.shape[0] * users.shape[0]

104927620180

In [10]:
# return the number of non-missing entries 
artist_user_plays

<292385x358868 sparse matrix of type '<class 'numpy.float32'>'
	with 17535606 stored elements in Compressed Sparse Row format>

In [ ]:
# investigate the proportion of non-zero entries
artist_user_plays.count_nonzero() / np.prod(artist_user_plays.shape) # non zero count / total count

0.0001671209636692248

## Preparing the data
Okapi BM25 (Best Matching) scoring is a ranking algorithm used by search engines to estimate the relevance of items to a given search query, based on the frequency of occurrences and the size of the reference pool.  The origin of the algorithm is used in search terms in a pool of documents.

For completeness, the BM25 score of query $Q=\{q_1, \ldots, q_n\}$ for a document $D$ is calculated as:

$$\text{BM25}(D, Q) = \sum_{i=1}^{n} \frac{IDF(q_i) \cdot f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{\text{avgD}})},$$
where
- $IDF(q_i)$ is the inverse document frequency of term $q_i$.
- $f(q_i, D)$ is the term frequency of $q_i$ in the document $D$.
- $k_1$ and $b$ are parameters controlling term saturation and document length normalization.
- $D$ is the length of the document.
- $\text{avgD}$ is the average document length in the corpus.


In [13]:
from implicit.nearest_neighbours import bm25_weight

# using the weighting function for normalization
artist_user_plays = bm25_weight(artist_user_plays, K1=100, B=0.8)
user_plays = artist_user_plays.T.tocsr()

## Training the model with alternating least squares

In [14]:
from implicit.als import AlternatingLeastSquares

# using alternating least squares algorithm
model = AlternatingLeastSquares(factors= 16, regularization = 0.5, alpha = 2.0)

model.fit(user_plays)

/opt/anaconda3/envs/cs326/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

## Similar artists recommendation

In [ ]:
# generate similar artist recommendation
list(artists).index('beyonce') # beyonce as example

45721

In [ ]:
artist_id = 45721 # beyonce id
ids, scores = model.similar_items(artist_id, N=10)

In [23]:
pd.DataFrame({'artist': artists[ids], 'score': scores})

,artist,score
0,beyonce,1.000000
1,ina,0.975915
2,akon feat. colby o donis and kardinal offishall,0.974678
3,cassie ft ryan leslie,0.973418
4,costi & alberto,0.972502
5,charlie wilson & pharrell,0.972290
6,akon feat. kat deluna,0.970691
7,shorti,0.969717
8,m. pokora,0.969336
9,ivena,0.967415


## User-specific recommendation

In [24]:
# generate user-based recommendation
userid = 10
ids, scores = model.recommend(userid, user_plays[userid], N=10)

In [25]:
artists[ids]

array(['johnossi', 'anna ternheim', 'moneybrother', 'hello saferide',
       'glasvegas', 'laakso', 'marit bergman', 'familjen',
       'broder daniel', 'billie the vision & the dancers'], dtype=object)